# Test Sealion

In [18]:
# Check free memory
print(f"GPU free: {torch.cuda.mem_get_info()[0]/1024**3:.2f} GB")
print(f"GPU total: {torch.cuda.mem_get_info()[1]/1024**3:.2f} GB")

GPU free: 0.41 GB
GPU total: 14.56 GB


In [2]:
!pip install transformers accelerate bitsandbytes torch pdfplumber -q

In [3]:
# Load SEA-LION with 4-bit quantization (fits in T4)

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_id = "aisingapore/llama3-8b-cpt-sea-lionv2.1-instruct"

# 4-bit quantization config — required for free T4 16GB
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": "cuda:0"}    # ← change "auto" to this, force all layers on GPU
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# If PDF, extract text first
import pdfplumber

with pdfplumber.open("/content/drive/MyDrive/AI-application-in-Law-Hackathon/data/VN_sample.pdf") as pdf:
    vn_text = "\n".join(page.extract_text() for page in pdf.pages if page.extract_text())

print(vn_text[:1000])  # preview first 1000 chars

CÔNG BÁO/Số 971 + 972/Ngày 24-7-2025 13
QUỐC HỘI CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM
Độc lập - Tự do - Hạnh phúc
Luật số: 91/2025/QH15
LUẬT
BẢO VỆ DỮ LIỆU CÁ NHÂN
Căn cứ Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam đã được sửa đổi,
bổ sung một số điều theo Nghị quyết số 203/2025/QH15;
Quốc hội ban hành Luật Bảo vệ dữ liệu cá nhân.
Chương I
NHỮNG QUY ĐỊNH CHUNG
Điều 1. Phạm vi điều chỉnh và đối tượng áp dụng
1. Luật này quy định về dữ liệu cá nhân, bảo vệ dữ liệu cá nhân và quyền,
nghĩa vụ, trách nhiệm của cơ quan, tổ chức, cá nhân có liên quan.
2. Luật này áp dụng đối với:
a) Cơ quan, tổ chức, cá nhân Việt Nam;
b) Cơ quan, tổ chức, cá nhân nước ngoài tại Việt Nam;
c) Cơ quan, tổ chức, cá nhân nước ngoài trực tiếp tham gia hoặc có liên quan
đến hoạt động xử lý dữ liệu cá nhân của công dân Việt Nam và người gốc Việt
Nam chưa xác định được quốc tịch đang sinh sống tại Việt Nam đã được cấp giấy
chứng nhận căn cước.
Điều 2. Giải thích từ ngữ
Trong Luật này, các từ ngữ dưới đây được hiểu

In [37]:
# Create translation function
def translate_vn_to_en_sea(vietnamese_text, max_new_tokens=2048):
    prompt = f"""<|im_start|>system
You are a legal document translator specializing in Southeast Asian law.
Translate ONLY the provided Vietnamese text to English.
Rules:
- Translate ALL text including headers, document references, and page numbers
- Preserve all article numbers, section references, and law codes exactly
- Keep proper nouns (organization names, law numbers) unchanged
- Do NOT add, infer, or generate content beyond what is given
- Preserve original formatting and line structure
<|im_end|>
<|im_start|>user
Translate this Vietnamese legal text to English:
{vietnamese_text}
<|im_end|>
<|im_start|>assistant
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            # temperature=0.1,       # low temperature = more deterministic, better for translation
            do_sample=False,
            repetition_penalty=1.1
        )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the assistant response
    return result.split("<|im_start|>assistant")[-1].strip()

In [38]:
# Test with one paragraph
sample_chunk = vn_text[:3000] # first 5000 chars

translated = translate_vn_to_en_sea(sample_chunk)
print("=== ORIGINAL ===")
print(sample_chunk)
print("\n=== TRANSLATED ===")
print(translated)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


=== ORIGINAL ===
CÔNG BÁO/Số 971 + 972/Ngày 24-7-2025 13
QUỐC HỘI CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM
Độc lập - Tự do - Hạnh phúc
Luật số: 91/2025/QH15
LUẬT
BẢO VỆ DỮ LIỆU CÁ NHÂN
Căn cứ Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam đã được sửa đổi,
bổ sung một số điều theo Nghị quyết số 203/2025/QH15;
Quốc hội ban hành Luật Bảo vệ dữ liệu cá nhân.
Chương I
NHỮNG QUY ĐỊNH CHUNG
Điều 1. Phạm vi điều chỉnh và đối tượng áp dụng
1. Luật này quy định về dữ liệu cá nhân, bảo vệ dữ liệu cá nhân và quyền,
nghĩa vụ, trách nhiệm của cơ quan, tổ chức, cá nhân có liên quan.
2. Luật này áp dụng đối với:
a) Cơ quan, tổ chức, cá nhân Việt Nam;
b) Cơ quan, tổ chức, cá nhân nước ngoài tại Việt Nam;
c) Cơ quan, tổ chức, cá nhân nước ngoài trực tiếp tham gia hoặc có liên quan
đến hoạt động xử lý dữ liệu cá nhân của công dân Việt Nam và người gốc Việt
Nam chưa xác định được quốc tịch đang sinh sống tại Việt Nam đã được cấp giấy
chứng nhận căn cước.
Điều 2. Giải thích từ ngữ
Trong Luật này, các từ ngữ d

In [8]:
# Translate → store with metadata
output = {
    "source_lang": "Vietnamese",
    "jurisdiction": "Vietnam",
    "law": "Law 91/2025/QH15 - Personal Data Protection",
    "original_text": vn_text[:500],
    "translated_text": translate_vn_to_en_sea(vn_text[:500]),
    "source_url": "VN_sample"
}

import json
print(json.dumps(output, indent=2, ensure_ascii=False))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


{
  "source_lang": "Vietnamese",
  "jurisdiction": "Vietnam",
  "law": "Law 91/2025/QH15 - Personal Data Protection",
  "original_text": "CÔNG BÁO/Số 971 + 972/Ngày 24-7-2025 13\nQUỐC HỘI CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM\nĐộc lập - Tự do - Hạnh phúc\nLuật số: 91/2025/QH15\nLUẬT\nBẢO VỆ DỮ LIỆU CÁ NHÂN\nCăn cứ Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam đã được sửa đổi,\nbổ sung một số điều theo Nghị quyết số 203/2025/QH15;\nQuốc hội ban hành Luật Bảo vệ dữ liệu cá nhân.\nChương I\nNHỮNG QUY ĐỊNH CHUNG\nĐiều 1. Phạm vi điều chỉnh và đối tượng áp dụng\n1. Luật này quy định về dữ liệu cá nhân, bảo vệ dữ liệu cá nhân và quyền,\nnghĩa vụ, t",
  "translated_text": "COUNCIL OF PEOPLE'S SUPREME COURT\nOF THE SOCIALIST REPUBLIC OF VIET NAM\nIndependence - Freedom - Happiness\nLaw No.: 91/2025/QH15\nLAW ON PROTECTION OF PERSONAL DATA\n\nPursuant to the amended and supplemented Constitution of the Socialist Republic of Vietnam by Resolution Number 203/2025/QH15;\n\nNational Assembly promu